# Drought simulation

Requires openpyxl.
Run beside monthly_rain_temp_v1.xlsx.

In [1]:
from openpyxl import load_workbook

book = load_workbook("monthly_rain_temp_v1.xlsx", data_only=True, read_only=True)
species = []
for row in book["variable values"].iter_rows(min_row=25, values_only=True):
    if row[0] == "Cedar Creek, MN" and row[1]:
        species.append((row[1], row[3], row[6]))

drought_scores = {}
for row in book["D(t)"].iter_rows(min_row=3, values_only=True):
    if isinstance(row[0], (int, float)):
        drought_scores[int(row[0])] = row[1]
book.close()

severe_year = max(drought_scores, key=drought_scores.get)
severe_drought = drought_scores[severe_year]
alpha = 1.0

# Diagonals describe growth; other entries describe facilitation.
interactions = []
for i in range(len(species)):
    coefficients = []
    for j in range(len(species)):
        if i == j:
            coefficient = species[i][1]
        else:
            coefficient = alpha * species[i][1] / (2 * (len(species) - 1)) # TODO: Come up with a better way to pick constants; right now we make the sum of all coefficients equal to the growth rate and consistent.
        coefficients.append(coefficient)
    interactions.append(coefficients)

print("Species:", len(species))
print("Interaction strength:", alpha)

Species: 16
Interaction strength: 1.0


## Model

Growth includes contributions from every community member.
Crowding multiplies growth; drought removes biomass.
Alpha controls facilitation; zero removes cross-species contributions.
Updates use previous biomass for every species.

In [2]:
def simulate(members, weather, steps_per_year=1000):
    # Every community starts with total biomass one.
    biomass = []
    for member in members:
        biomass.append(1.0 / len(members))

    totals = [sum(biomass)]
    dt = 1.0 / steps_per_year

    for drought in weather:
        for step in range(steps_per_year):
            total = sum(biomass)
            updated = []
            for position in range(len(members)):
                member = members[position]
                name, growth, sensitivity = species[member]
                amount = biomass[position]
                growth_input = 0.0
                for other_position in range(len(members)):
                    other = members[other_position]
                    growth_input += interactions[member][other] * biomass[other_position]
                change = growth_input * (1 - total)
                change -= sensitivity * amount * drought
                new_amount = amount + dt * change
                if new_amount < 0:
                    raise ValueError("Reduce the timestep: negative biomass.")
                updated.append(new_amount)
            biomass = updated
        totals.append(sum(biomass))

    return totals

## Experiment 1: one severe year

Compare the mixture against mean and best monocultures.
All communities start with total biomass one.

In [3]:
monoculture_results = []
for member in range(len(species)):
    totals = simulate([member], [severe_drought])
    monoculture_results.append(totals[-1])

all_members = list(range(len(species)))
mixture = simulate(all_members, [severe_drought])[-1]
baseline = sum(monoculture_results) / len(monoculture_results)
best = max(monoculture_results)
gain = 100 * (mixture - baseline)

print("Severe drought source year:", severe_year)
print(f"Mean monoculture: {100 * baseline:.2f}%")
print(f"Best monoculture: {100 * best:.2f}%")
print(f"Full mixture:    {100 * mixture:.2f}%")
print(f"Mixture gain:    {gain:.2f} percentage points")

# Compare timestep sizes for this mixture.
finer = simulate(all_members, [severe_drought], 2000)[-1]
print(f"Timestep difference: {100 * abs(finer - mixture):.4f} points")

Severe drought source year: 1976
Mean monoculture: 72.98%
Best monoculture: 79.49%
Full mixture:    81.13%
Mixture gain:    8.15 percentage points
Timestep difference: 0.0016 points


## Experiment 2: historical weather

Biomass carries forward through 1987–1991.
Weather stays constant within each year.

In [4]:
years = list(range(1987, 1992))
weather = []
for year in years:
    weather.append(drought_scores[year])

totals = simulate(all_members, weather)
control = simulate(all_members, [0.0] * len(years))
print("Year    Drought    Mixture    Control")
for position in range(len(years)):
    year = years[position]
    drought = weather[position]
    total = totals[position + 1]
    normal = control[position + 1]
    print(f"{year}    {drought:7.3f}    {total:7.3f}    {normal:7.3f}")

Year    Drought    Mixture    Control
1987      0.557      0.869      1.000
1988      0.608      0.858      1.000
1989      0.228      0.945      1.000
1990     -0.187      1.042      1.000
1991     -0.482      1.114      1.000


## Experiment 3: Vary species count (todo)

Save final biomass for each species count.
Use the existing interaction matrix and severe drought.
Species identity changes alongside species count.

In [5]:
richness_results = []
for size in range(1, len(species) + 1):
    members = list(range(size))
    pass